# Stage 1: Select Implied Copyable Trades

Find profitable follower BUYs that follow a leader's BUY or SELL on the same
token within a time window. Two separate leader groups: **buy leaders** (whose
BUYs precede follower BUYs) and **sell leaders** (whose SELLs precede follower
BUYs).

Grid-search over selection thresholds to maximize **copyable PnL from implied
trades** on the validation split.

**Output:** `stage1_implied_result.json` with best selection params.

In [39]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd

from lib import (
    load_trades,
    split_data,
    compute_copyable_notional,
    compute_opening_metrics,
    select_follower_wallets,
    select_leader_wallets,
    detect_implied_buys,
    score_leaders,
    evaluate_implied_pnl,
    evaluate_follower_buy_performance,
    iterative_leader_follower_filter,
    filter_stable_leaders,
    filter_leaders_by_drawdown,
    filter_followers_by_drawdown,
    filter_followers_by_val_roi,
    filter_pairs_by_frequency,
    run_implied_grid_search,
    DEFAULT_TAGS
)
from polymarket_analysis.wallet_selection.volatility import compute_wallet_metrics


pd.options.display.float_format = "{:.4f}".format
pd.options.display.max_rows = 100

print(f"Tags: {DEFAULT_TAGS}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Tags: {'Weather'}


## Parameters

In [40]:
if DEFAULT_TAGS == {"Weather"}:
    PARAMS = dict(
        # --- Follower selection ---
        min_follower_copyable_roi=0.05,
        min_follower_trade_value=100,
        min_follower_num_buckets=30,
        max_follower_hhi=0.3,
        # --- Leader selection ---
        min_leader_trade_count=20,
        max_leader_hhi=1,
        # --- Broad follower set for leader stability detection ---
        stability_min_follower_roi=0.0,
        stability_min_follower_buckets=10,
        stability_max_follower_hhi=1,
        # --- Leader filtering (drawdown-based) ---
        max_dd_pnl_ratio=0.3,
        # --- Follower filtering (drawdown-based) ---
        follower_max_dd_pnl_ratio=0.3,
        # --- Pair filtering ---
        min_pair_observations=3,
        # --- Follower filtering (val ROI) ---
        min_val_roi=0.07,
        # --- Time window ---
        time_window_minutes=15,
    )
elif DEFAULT_TAGS == {"Politics"}:
    PARAMS = dict(
        # --- Follower selection ---
        min_follower_copyable_roi=0.30,
        min_follower_trade_value=100,
        min_follower_num_buckets=30,
        max_follower_hhi=0.3,
        # --- Leader selection ---
        min_leader_trade_count=20,
        max_leader_hhi=1,
        # --- Broad follower set for leader stability detection ---
        stability_min_follower_roi=0.0,
        stability_min_follower_buckets=10,
        stability_max_follower_hhi=1,
        # --- Stability filtering ---
        stability_n_splits=3,
        stability_min_profitable_splits=2,
        # --- Follower filtering (drawdown-based) ---
        follower_max_dd_pnl_ratio=0.3,
        # --- Pair filtering ---
        min_pair_observations=3,
        # --- Follower filtering (val ROI) ---
        min_val_roi=0.07,
        # --- Time window ---
        time_window_minutes=30,
    )
else:
    raise ValueError(f"Unsupported DEFAULT_TAGS: {DEFAULT_TAGS}")

for k, v in PARAMS.items():
    print(f"  {k}: {v}")

  min_follower_copyable_roi: 0.05
  min_follower_trade_value: 100
  min_follower_num_buckets: 30
  max_follower_hhi: 0.3
  min_leader_trade_count: 20
  max_leader_hhi: 1
  stability_min_follower_roi: 0.0
  stability_min_follower_buckets: 10
  stability_max_follower_hhi: 1
  max_dd_pnl_ratio: 0.3
  follower_max_dd_pnl_ratio: 0.3
  min_pair_observations: 3
  min_val_roi: 0.07
  time_window_minutes: 15


## Load data

In [56]:
df_full = load_trades()
df_full = compute_copyable_notional(df_full)

train_cutoff = pd.Timestamp("2026-06-01", tz="UTC")
test_cutoff = pd.Timestamp("2026-07-01", tz="UTC")

df_train, df_val, df_test = split_data(df_full, method='chronological')

Markets: 1974837
Filtered markets for {'Weather'}: 91123
Loading 16 trade shards...
Total trades loaded: 14,250,603
Unique wallets: 4,082
Date range: 2025-01-09 15:32:39+00:00 -> 2026-07-27 06:12:25+00:00
Chronological split: train <= 2026-05-21T00:00:00Z, val <= 2026-06-23T00:00:00Z, test > 2026-06-23T00:00:00Z
Method: chronological  |  Unique end dates: 112  (train=44, val=33, test=35)

  Train:  4,337,961 trades  (15,923 markets)
  Val:    5,211,194 trades  (20,297 markets)
  Test:   4,701,448 trades  (20,655 markets)
  Total: 14,250,603 trades  (56,875 markets)


## Compute wallet metrics on training data

In [42]:
wallet_vol, _ = compute_wallet_metrics(df_train)

wallet_vol["copyable_pnl_factor"] = np.clip(
    wallet_vol["copyable_pnl"] / wallet_vol["total_pnl"].replace(0, np.nan),
    0, 1.0,
).fillna(0.0)
wallet_vol["copyable_roi"] = wallet_vol["average_roi"] * wallet_vol["copyable_pnl_factor"]

opening_metrics = compute_opening_metrics(df_train)
wallet_vol = wallet_vol.merge(opening_metrics, on="wallet", how="left")
for c in ["opening_roi", "opening_pnl", "opening_copyable_roi", "opening_copyable_pnl"]:
    wallet_vol[c] = wallet_vol[c].fillna(0.0)

print(f"Wallets with metrics: {len(wallet_vol)}")
wallet_vol[["wallet", "buy_roi", "sell_roi", "copyable_pnl", "copyable_roi", "num_buckets"]].head(10)

Wallets with metrics: 3220


,wallet,buy_roi,sell_roi,copyable_pnl,copyable_roi,num_buckets
0,0x0054ee7dfb882d2d016fa13ef5f5cdb3b0ebcf1f,0.0088,0.0012,-0.5270,-0.0000,38
1,0x005ed998fcb786679eb8bfd0d20c15c0903d6d8e,0.0394,0.1396,2.5901,0.0696,208
2,0x00833cc2d777e6f2fc8437679124024ae6468cb1,-0.0700,0.0500,0.0000,0.0000,2
3,0x0141be702d272f17666e280303ad44e7bc0cc2da,0.0288,-0.0100,57.7247,0.2432,34
4,0x015be8bad14c79d2722a0bd8bbe0cd93b905556d,-0.0094,-0.0250,-17.5761,-0.2197,299
5,0x01a5fb1fa13f378138a31382c8364b5d4e2b0e36,0.0084,-0.0016,3.2916,-0.0469,23
6,0x01a68281185e728ba0fef6245008bf8af68a59b0,0.0099,-0.1444,-0.6680,0.0000,844
7,0x01ced860d8dca5d7987579d2a2635df8520d27a2,0.0726,-0.0402,-401.0333,0.0000,989
8,0x01d94480e2a96cdd01fed071878b1adf82e0acd0,0.0001,-0.0009,30.6519,0.0000,1174
9,0x01f2a8baabe17c2541d1e3091220991f257ac3de,0.0051,0.0034,-3381.9584,0.0000,39804


## Baseline selection

In [43]:
follower_wallets = select_follower_wallets(
    wallet_vol,
    min_copyable_roi=PARAMS["min_follower_copyable_roi"],
    min_trade_value=PARAMS["min_follower_trade_value"],
    min_num_buckets=PARAMS["min_follower_num_buckets"],
    max_market_pnl_hhi=PARAMS["max_follower_hhi"],
)
print(f"Followers: {len(follower_wallets)}")

buy_leaders = select_leader_wallets(
    wallet_vol,
    min_trade_count=PARAMS["min_leader_trade_count"],
    min_roi=None,
    max_market_pnl_hhi=PARAMS["max_leader_hhi"],
    side="BUY",
)
print(f"Buy leaders: {len(buy_leaders)}")

sell_leaders = select_leader_wallets(
    wallet_vol,
    min_trade_count=PARAMS["min_leader_trade_count"],
    min_roi=None,
    max_market_pnl_hhi=PARAMS["max_leader_hhi"],
    side="SELL",
)
print(f"Sell leaders: {len(sell_leaders)}")

Followers: 110
Buy leaders: 1216
Sell leaders: 1216


## Baseline evaluation

In [44]:
follower_ws = set(follower_wallets['wallet'])
buy_leader_ws = set(buy_leaders['wallet'])
sell_leader_ws = set(sell_leaders['wallet'])
tw = PARAMS["time_window_minutes"]

for split_name, df_split in  [("TRAIN", df_train), ("VAL", df_val), ("TEST", df_test)]:
    buy_ev = evaluate_implied_pnl(
        df_split, follower_ws, buy_leader_ws,
        time_window_minutes=tw, leader_side="BUY"
    )
    sell_ev = evaluate_implied_pnl(
        df_split, follower_ws, sell_leader_ws,
        time_window_minutes=tw, leader_side="SELL",
    )
    total = buy_ev["followed_copyable_pnl"] + sell_ev["followed_copyable_pnl"]
    notional = buy_ev["followed_copyable_notional"] + sell_ev["followed_copyable_notional"]
    roi = total / notional if notional > 0 else 0
    print(f"{split_name}: buy_pnl={buy_ev['followed_copyable_pnl']:.2f} ({buy_ev['trade_count']} trades, {buy_ev['leader_count']} leaders)  "
          f"sell_pnl={sell_ev['followed_copyable_pnl']:.2f} ({sell_ev['trade_count']} trades, {sell_ev['leader_count']} leaders)  "
          f"total={total:.2f}  roi={roi:.4f}")

TRAIN: buy_pnl=30437.11 (27314 trades, 874 leaders)  sell_pnl=34951.14 (33755 trades, 611 leaders)  total=65388.25  roi=0.1401
VAL: buy_pnl=6581.19 (112891 trades, 752 leaders)  sell_pnl=4000.76 (91164 trades, 590 leaders)  total=10581.95  roi=0.0123
TEST: buy_pnl=4129.80 (81599 trades, 564 leaders)  sell_pnl=8866.79 (62372 trades, 448 leaders)  total=12996.60  roi=0.0216


## Score leaders (baseline)

In [45]:
# Buy leaders
buy_implied = detect_implied_buys(
    df_train, follower_ws, buy_leader_ws,
    time_window_minutes=tw, leader_side="BUY",
)
buy_scores = score_leaders(buy_implied)
print("Top buy leaders:")
print(buy_scores.head(10).to_string())

print()

# Sell leaders
sell_implied = detect_implied_buys(
    df_train, follower_ws, sell_leader_ws,
    time_window_minutes=tw, leader_side="SELL",
)
sell_scores = score_leaders(sell_implied)
print("Top sell leaders:")
print(sell_scores.head(10).to_string())

Top buy leaders:
                                leader_wallet  num_followers  total_follower_copyable_pnl  num_followed_trades  unique_tokens
0  0x77bdeb3f229bf6d826d20dbd5f0c7972c32ae48f             81                    1563.0922                 1152            403
1  0xb06a0eae498750ed0acac7e1f759f741c56e52f5             95                    1482.7761                 1246            766
2  0x510f4963b66b1b18505faab74b0bb943d1dda43c             90                    1270.1113                  679            398
3  0xb40e89677d59665d5188541ad860450a6e2a7cc9             98                    1169.5210                 1266            765
4  0x9cb35555b913c805a62a2e34bce4ef14f0e81367             87                     893.8172                  675            376
5  0x570c03b5f037294ca24b743d90df8c1a993f91c5             49                     874.7689                  188            120
6  0xf13242c0f5ed15fe9aec0c5eab40a2a8dbafc706             28                     849.5647            

## Improved pipeline: Stable leaders + followers + pair filtering

Filter leaders who are consistently profitable across training time slices,
filter followers whose implied PnL has low drawdown, then require a minimum
number of observed copy-trades per leader-follower pair.

In [46]:
# Broad follower set for leader stability detection (more implied trades per leader)
fw_broad = set(select_follower_wallets(
    wallet_vol,
    min_copyable_roi=PARAMS["stability_min_follower_roi"],
    min_trade_value=PARAMS["min_follower_trade_value"],
    min_num_buckets=PARAMS["stability_min_follower_buckets"],
    max_market_pnl_hhi=PARAMS["stability_max_follower_hhi"],
)["wallet"])

if DEFAULT_TAGS == {"Weather"}:
    stable_buy_leaders = filter_leaders_by_drawdown(
        df_train, buy_leader_ws, fw_broad,
        time_window_minutes=tw, leader_side="BUY",
        max_dd_pnl_ratio=PARAMS["max_dd_pnl_ratio"],
    )
    stable_sell_leaders = filter_leaders_by_drawdown(
        df_train, sell_leader_ws, fw_broad,
        time_window_minutes=tw, leader_side="SELL",
        max_dd_pnl_ratio=PARAMS["max_dd_pnl_ratio"],
    )
elif DEFAULT_TAGS == {"Politics"}:
    stable_buy_leaders = filter_stable_leaders(
        df_train, buy_leader_ws, fw_broad,
        time_window_minutes=tw, leader_side="BUY",
        n_splits=PARAMS["stability_n_splits"],
        min_profitable_splits=PARAMS["stability_min_profitable_splits"],
    )
    stable_sell_leaders = filter_stable_leaders(
        df_train, sell_leader_ws, fw_broad,
        time_window_minutes=tw, leader_side="SELL",
        n_splits=PARAMS["stability_n_splits"],
        min_profitable_splits=PARAMS["stability_min_profitable_splits"],
    )
else:
    raise ValueError(f"Unsupported DEFAULT_TAGS: {DEFAULT_TAGS}")

print(f"Stable buy leaders: {len(stable_buy_leaders)} / {len(buy_leader_ws)}")
print(f"Stable sell leaders: {len(stable_sell_leaders)} / {len(sell_leader_ws)}")

buy_imp_train = detect_implied_buys(
    df_train, follower_ws, stable_buy_leaders,
    time_window_minutes=tw, leader_side="BUY",
)
sell_imp_train = detect_implied_buys(
    df_train, follower_ws, stable_sell_leaders,
    time_window_minutes=tw, leader_side="SELL",
)

min_obs = PARAMS["min_pair_observations"]
buy_pairs = filter_pairs_by_frequency(buy_imp_train, min_observations=min_obs)
sell_pairs = filter_pairs_by_frequency(sell_imp_train, min_observations=min_obs)

final_buy_leaders = set(buy_pairs["leader_wallet"]) if not buy_pairs.empty else set()
final_sell_leaders = set(sell_pairs["leader_wallet"]) if not sell_pairs.empty else set()
final_followers = (
    (set(buy_pairs["follower_wallet"]) if not buy_pairs.empty else set())
    | (set(sell_pairs["follower_wallet"]) if not sell_pairs.empty else set())
)

# Filter followers by drawdown on TRAIN+VAL combined implied trades
df_train_val = pd.concat([df_train, df_val], ignore_index=True)
buy_imp_train_val = detect_implied_buys(
    df_train_val, follower_ws, stable_buy_leaders,
    time_window_minutes=tw, leader_side="BUY",
)
sell_imp_train_val = detect_implied_buys(
    df_train_val, follower_ws, stable_sell_leaders,
    time_window_minutes=tw, leader_side="SELL",
)
follower_dd_ratio = PARAMS.get("follower_max_dd_pnl_ratio")
stable_followers = filter_followers_by_drawdown(
    buy_imp_train_val, sell_imp_train_val,
    max_dd_pnl_ratio=follower_dd_ratio,
)
removed_by_dd = final_followers - stable_followers
final_followers = final_followers & stable_followers

# Filter followers by VAL ROI (generalization check)
min_val_roi = PARAMS["min_val_roi"]
val_roi_followers = filter_followers_by_val_roi(
    df_val, final_followers, final_buy_leaders, final_sell_leaders,
    time_window_minutes=tw, min_val_roi=min_val_roi,
)
removed_by_val = final_followers - val_roi_followers
final_followers = final_followers & val_roi_followers

print()  # newline before summary
print(f"After pair filter (min_obs={min_obs}):")
print(f"  Followers: {len(final_followers)} (dd removed {len(removed_by_dd)}, val_roi removed {len(removed_by_val)})")
print(f"  Buy leaders: {len(final_buy_leaders)}")
print(f"  Sell leaders: {len(final_sell_leaders)}")

Stable buy leaders: 484 / 1216
Stable sell leaders: 355 / 1216

After pair filter (min_obs=3):
  Followers: 19 (dd removed 70, val_roi removed 18)
  Buy leaders: 216
  Sell leaders: 124


In [47]:
print("=" * 70)
print(f"EVALUATION (tw={tw}min, min_pair_obs={min_obs})")
print("=" * 70)

for split_name, df_split in [("TRAIN", df_train), ("VAL", df_val), ("TEST", df_test)]:
    buy_ev = evaluate_implied_pnl(df_split, final_followers, final_buy_leaders, time_window_minutes=tw, leader_side="BUY")
    sell_ev = evaluate_implied_pnl(df_split, final_followers, final_sell_leaders, time_window_minutes=tw, leader_side="SELL")
    follower_buy = evaluate_follower_buy_performance(df_split, final_followers)

    n_active = len(set(df_split[df_split["wallet"].isin(final_followers)]["wallet"]))
    n_markets = df_split["condition_id"].nunique()

    imp_pnl = buy_ev["followed_copyable_pnl"] + sell_ev["followed_copyable_pnl"]
    imp_notional = buy_ev["followed_copyable_notional"] + sell_ev["followed_copyable_notional"]
    imp_trades = buy_ev["trade_count"] + sell_ev["trade_count"]
    imp_roi = imp_pnl / imp_notional if imp_notional > 0 else 0.0

    b_roi = buy_ev["followed_copyable_pnl"] / buy_ev["followed_copyable_notional"] if buy_ev["followed_copyable_notional"] > 0 else 0.0
    s_roi = sell_ev["followed_copyable_pnl"] / sell_ev["followed_copyable_notional"] if sell_ev["followed_copyable_notional"] > 0 else 0.0

    print(f"\n{split_name} ({n_markets} markets, {n_active} active followers):")
    print(f"  Implied BUY:   Copyable PnL: {buy_ev['followed_copyable_pnl']:>10.2f}  ROI: {b_roi:>7.4f}  ({buy_ev['trade_count']} trades, {buy_ev['leader_count']} leaders)")
    print(f"  Implied SELL:  Copyable PnL: {sell_ev['followed_copyable_pnl']:>10.2f}  ROI: {s_roi:>7.4f}  ({sell_ev['trade_count']} trades, {sell_ev['leader_count']} leaders)")
    print(f"  Implied total: Copyable PnL: {imp_pnl:>10.2f}  ROI: {imp_roi:>7.4f}  ({imp_trades} trades)")
    print(f"  All buys:      Copyable PnL: {follower_buy['followed_copyable_pnl']:>10.2f}  ROI: {follower_buy['followed_copyable_roi']:>7.4f}  (wallet PnL: {follower_buy['wallet_pnl']:>10.2f}, {follower_buy['trade_count']} trades)")

EVALUATION (tw=15min, min_pair_obs=3)

TRAIN (15923 markets, 19 active followers):
  Implied BUY:   Copyable PnL:   11183.83  ROI:  0.2705  (6198 trades, 167 leaders)
  Implied SELL:  Copyable PnL:   10923.73  ROI:  0.1728  (7304 trades, 100 leaders)
  Implied total: Copyable PnL:   22107.56  ROI:  0.2114  (13502 trades)
  All buys:      Copyable PnL:   11827.21  ROI:  0.1487  (wallet PnL:   37175.77, 12975 trades)

VAL (20297 markets, 19 active followers):
  Implied BUY:   Copyable PnL:    9658.86  ROI:  0.2650  (5956 trades, 125 leaders)
  Implied SELL:  Copyable PnL:   10686.67  ROI:  0.2367  (7234 trades, 76 leaders)
  Implied total: Copyable PnL:   20345.53  ROI:  0.2493  (13190 trades)
  All buys:      Copyable PnL:   12507.86  ROI:  0.1919  (wallet PnL:   52500.28, 13806 trades)

TEST (20655 markets, 15 active followers):
  Implied BUY:   Copyable PnL:    2921.95  ROI:  0.1059  (3976 trades, 86 leaders)
  Implied SELL:  Copyable PnL:    3677.89  ROI:  0.1184  (4935 trades, 51 le

## Summary

In [48]:
# Store for save cell
b_fw = final_followers
b_blw = final_buy_leaders
b_slw = final_sell_leaders

print(f"Final wallet counts:")
print(f"  Followers: {len(b_fw)}")
print(f"  Buy leaders: {len(b_blw)}")
print(f"  Sell leaders: {len(b_slw)}")

Final wallet counts:
  Followers: 19
  Buy leaders: 216
  Sell leaders: 124


In [49]:
best_params = PARAMS.copy()
print("Pipeline params:")
for k, v in best_params.items():
    print(f"  {k}: {v}")

Pipeline params:
  min_follower_copyable_roi: 0.05
  min_follower_trade_value: 100
  min_follower_num_buckets: 30
  max_follower_hhi: 0.3
  min_leader_trade_count: 20
  max_leader_hhi: 1
  stability_min_follower_roi: 0.0
  stability_min_follower_buckets: 10
  stability_max_follower_hhi: 1
  max_dd_pnl_ratio: 0.3
  follower_max_dd_pnl_ratio: 0.3
  min_pair_observations: 3
  min_val_roi: 0.07
  time_window_minutes: 15


## Concentration diagnostics (test split)

In [50]:
buy_imp_test = detect_implied_buys(df_test, b_fw, b_blw, time_window_minutes=tw, leader_side="BUY")
sell_imp_test = detect_implied_buys(df_test, b_fw, b_slw, time_window_minutes=tw, leader_side="SELL")
imp_test = pd.concat([buy_imp_test, sell_imp_test], ignore_index=True)

if imp_test.empty:
    print("No implied trades on test set.")
else:
    total_pnl = imp_test["copyable_pnl"].sum()
    total_trades = len(imp_test)
    print(f"Test implied trades: {total_trades:,}   Total copyable PnL: ${total_pnl:,.2f}\n")

    # --- Leader concentration ---
    leader_pnl = (
        imp_test.groupby("leader_wallet", sort=False)["copyable_pnl"]
        .agg(["sum", "count", "nunique"])
        .rename(columns={"sum": "pnl", "count": "trades", "nunique": "followers"})
        .sort_values("pnl", ascending=False)
    )
    leader_pnl["cum_pnl"] = leader_pnl["pnl"].cumsum()
    leader_pnl["cum_pct"] = leader_pnl["cum_pnl"] / total_pnl
    n_leaders = len(leader_pnl)
    print(f"Leaders contributing to test PnL: {n_leaders}")
    for k in [1, 3, 5, 10]:
        if k <= n_leaders:
            pct = leader_pnl.iloc[k - 1]["cum_pct"]
            print(f"  Top {k:>2} leader(s): ${leader_pnl.iloc[k - 1]['cum_pnl']:>10,.2f}  ({pct:.1%} of total)")
    print()
    print("Top 10 leaders:")
    print(leader_pnl.head(10).to_string())

    # --- Follower concentration ---
    follower_pnl = (
        imp_test.groupby("follower_wallet", sort=False)["copyable_pnl"]
        .agg(["sum", "count"])
        .rename(columns={"sum": "pnl", "count": "trades"})
        .sort_values("pnl", ascending=False)
    )
    follower_pnl["cum_pnl"] = follower_pnl["pnl"].cumsum()
    follower_pnl["cum_pct"] = follower_pnl["cum_pnl"] / total_pnl
    n_followers = len(follower_pnl)
    print(f"\nFollowers active on test: {n_followers}")
    for k in [1, 5, 10, 20]:
        if k <= n_followers:
            pct = follower_pnl.iloc[k - 1]["cum_pct"]
            print(f"  Top {k:>2} follower(s): ${follower_pnl.iloc[k - 1]['cum_pnl']:>10,.2f}  ({pct:.1%} of total)")
    print()
    print("Top 10 followers:")
    print(follower_pnl.head(10).to_string())

    # --- Market concentration ---
    market_pnl = (
        imp_test.groupby("condition_id", sort=False)["copyable_pnl"]
        .agg(["sum", "count"])
        .rename(columns={"sum": "pnl", "count": "trades"})
        .sort_values("pnl", ascending=False)
    )
    market_pnl["cum_pnl"] = market_pnl["pnl"].cumsum()
    market_pnl["cum_pct"] = market_pnl["cum_pnl"] / total_pnl
    n_markets = len(market_pnl)
    print(f"\nMarkets with implied trades: {n_markets}")
    for k in [1, 3, 5, 10]:
        if k <= n_markets:
            pct = market_pnl.iloc[k - 1]["cum_pct"]
            print(f"  Top {k:>2} market(s):  ${market_pnl.iloc[k - 1]['cum_pnl']:>10,.2f}  ({pct:.1%} of total)")
    print()
    print("Top 10 markets:")
    print(market_pnl.head(10).to_string())

    # --- Negative PnL followers ---
    neg_followers = (follower_pnl["pnl"] < 0).sum()
    neg_pnl = follower_pnl.loc[follower_pnl["pnl"] < 0, "pnl"].sum()
    print(f"\nFollowers with negative PnL: {neg_followers}  (total: ${neg_pnl:,.2f})")
    pos_followers = (follower_pnl["pnl"] > 0).sum()
    pos_pnl = follower_pnl.loc[follower_pnl["pnl"] > 0, "pnl"].sum()
    print(f"Followers with positive PnL: {pos_followers}  (total: ${pos_pnl:,.2f})")

    # --- Gini coefficient on leader PnL ---
    vals = leader_pnl["pnl"].values
    vals_sorted = np.sort(vals)
    n = len(vals_sorted)
    cum = np.cumsum(vals_sorted)
    gini = 1 - 2 * np.sum(cum) / (n * cum[-1]) if cum[-1] > 0 else 0.0
    print(f"\nLeader PnL Gini coefficient: {gini:.4f}  (1 = perfect concentration, 0 = equal)")

Test implied trades: 8,911   Total copyable PnL: $6,599.84

Leaders contributing to test PnL: 119
  Top  1 leader(s): $  2,114.56  (32.0% of total)
  Top  3 leader(s): $  4,000.39  (60.6% of total)
  Top  5 leader(s): $  4,875.43  (73.9% of total)
  Top 10 leader(s): $  5,760.75  (87.3% of total)

Top 10 leaders:
                                                 pnl  trades  followers   cum_pnl  cum_pct
leader_wallet                                                                             
0x945a49252f772a10c6ddd1d1e1e24ee20438a48c 2114.5619    3649       2050 2114.5619   0.3204
0xb40e89677d59665d5188541ad860450a6e2a7cc9 1134.0497    2026       1326 3248.6117   0.4922
0x6243d0b16c5030dbddaa5a51d9134337d1c3e014  751.7764     121         87 4000.3881   0.6061
0x26123cbf0f4820f7e70408a8c054ba7615c05289  558.2825     367        233 4558.6706   0.6907
0xc34f6b088bb9172625ee1ea2ee8da9ac4f037d2e  316.7571     223        162 4875.4277   0.7387
0x686880eea810fa9141be46a1acec9eee41755198  275.

## Save stage 1 result

In [51]:
import json
from datetime import datetime, timezone
from pathlib import Path


def _convert(obj):
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj


# Collect wallet records for each group
wallet_cols = [
    "wallet", "buy_roi", "sell_roi", "copyable_pnl", "copyable_roi",
    "num_buckets", "num_markets", "total_notional", "total_pnl",
    "market_pnl_hhi",
]

def _wallet_records(df):
    if df is None or df.empty:
        return []
    cols = [c for c in wallet_cols if c in df.columns]
    records = df[cols].to_dict(orient="records")
    return [{k: _convert(v) for k, v in w.items()} for w in records]


metadata = {
    "type": "implied",
    "tags": sorted(DEFAULT_TAGS),
    "run_timestamp": datetime.now(timezone.utc).isoformat(),
    "n_followers": len(b_fw),
    "n_buy_leaders": len(b_blw),
    "n_sell_leaders": len(b_slw),
    "n_wallets_total": len(wallet_vol),
}

payload = {
    "stage": 1,
    "type": "implied",
    "best_params": {k: _convert(v) for k, v in best_params.items()},
    "metadata": metadata,
    "wallets": {
        "followers": _wallet_records(follower_wallets[follower_wallets["wallet"].isin(b_fw)]),
        "buy_leaders": _wallet_records(buy_leaders[buy_leaders["wallet"].isin(b_blw)]),
        "sell_leaders": _wallet_records(sell_leaders[sell_leaders["wallet"].isin(b_slw)]),
    },
}

out_path = Path("stage1_implied_result.json")
with open(out_path, "w") as f:
    json.dump(payload, f, indent=2)
print(f"Saved stage 1 implied result -> {out_path.resolve()}")

Saved stage 1 implied result -> /Users/vobornij/projects/polymarket/notebooks/wallet_selection/stage1_implied_result.json


In [52]:
df_test.columns

Index(['wallet', 'condition_id', 'token_id', 'dt', 'side', 'position',
       'quantity', 'price', 'usdc_amount', 'final_value_usdc', 'trade_pnl',
       'copyable_pnl', 'token_winner', 'final_price',
       'last_condition_trade_ts', 'tx_hash', 'num_fills', 'is_train',
       'copyable_qty', 'avail_copy_total_vol', 'avail_copy_count',
       'end_date_iso', 'question', 'tags', 'primary_tag', 'winner_token_id',
       'outcome', 'pnl', 'notional', 'copyable_notional', 'roi',
       'copyable_roi'],
      dtype='object')

In [53]:
df_test[
    (df_test['condition_id'] == '0x18d19ee1593c5f758c9b652fae58b0ea98e8b28f3cc6c6ce42816dd032f9c8a7')
    & (df_test['dt'] >= pd.Timestamp("2026-07-09 07:37:31+00:00", tz="UTC"))
    & (df_test['pnl'] >= 0)
    & (df_test['dt'] <= pd.Timestamp("2026-07-09 07:42:31+00:00", tz="UTC"))
    # & (df_test['price'] <= 0.149)
       ].sort_values("dt")[['dt', 'wallet', 'side', 'price', 'trade_pnl', 'copyable_pnl', 'token_id', 'tx_hash']]

,dt,wallet,side,price,trade_pnl,copyable_pnl,token_id,tx_hash
1352015,2026-07-09 07:37:31+00:00,0x945a49252f772a10c6ddd1d1e1e24ee20438a48c,SELL,0.9490,2.7726,0.0000,2254165225066728302189294187670622099851703025...,0x47068d48c7375880cf06fb2eed3ec891e1fa31922caa...
1352016,2026-07-09 07:37:31+00:00,0x945a49252f772a10c6ddd1d1e1e24ee20438a48c,BUY,0.0510,1.2174,0.0000,3120308255357225723884420077341617187891774522...,0x47068d48c7375880cf06fb2eed3ec891e1fa31922caa...
1411723,2026-07-09 07:37:31+00:00,0xafde461fce5aa0fabdb7711c59db93b65e343e1d,SELL,0.8656,164.3030,163.4001,2254165225066728302189294187670622099851703025...,0x4f43e1da2143ef966eb26aa4bf77c759d42a1dfc5a70...
1411724,2026-07-09 07:37:31+00:00,0xafde461fce5aa0fabdb7711c59db93b65e343e1d,BUY,0.1496,600.5540,187.7898,3120308255357225723884420077341617187891774522...,0x4f43e1da2143ef966eb26aa4bf77c759d42a1dfc5a70...
908618,2026-07-09 07:37:38+00:00,0x0a8eee5cb1f039540abbbecc03571146048c05a4,BUY,0.1600,72.9708,72.9708,3120308255357225723884420077341617187891774522...,0x19f2f7c81255c35a80a79731b28576c19e10798973bf...
1292515,2026-07-09 07:37:38+00:00,0x8f514910c5e1f2ce541d01d8bbd8e9bea9638b09,BUY,0.1967,20.3806,20.3806,3120308255357225723884420077341617187891774522...,0x0bb56bf4f13cc4d1318f2841a7631cef4fe647560698...
1411725,2026-07-09 07:37:52+00:00,0xafde461fce5aa0fabdb7711c59db93b65e343e1d,BUY,0.1505,88.2706,87.7934,3120308255357225723884420077341617187891774522...,0xf0dac8e4bcb002f490bc07246233116dc6a39b15d440...


In [54]:
buy_imp_test.groupby('condition_id').agg(
    num_trades=('copyable_pnl', 'count'),
    total_copyable_pnl=('copyable_pnl', 'sum'),
    total_copyable_notional=('copyable_notional', 'sum')
).sort_values(by='total_copyable_pnl', ascending=False).head(10)

,num_trades,total_copyable_pnl,total_copyable_notional
condition_id,,,
0x1a15b9db7aa08ff03f4c15471c8f6c404293c69b9d466b86d26042b24cbfb279,3,690.6015,68.9369
0xda1ccda86cb6e17f952b744fc859c6c222a1054e5288e4750493555370d3d5e9,15,268.9522,585.5953
0x649e4e04f648c4229925590ab7bc114bc1bbe08bd3de0253110d528a2eef6b01,19,239.6498,69.7050
0xbfe7ef7a30c2bd848216a29186c97504dda0796112c9c5e6b8beb8ba162f8c1b,6,221.9400,30.0600
0x70a3a63367d470ecf8a23e6bade60469afefd57922c720e57c8345e04769b11a,22,212.7059,314.9766
0x74ca7aa98d9112934b6dcfc485c12eaedaa582eb4c1f1502611c030313625ffa,1,135.5556,32.4444
0x70a1d9a125b00ee96a95cfaecd18142e83a20b254d182d838fdb2d49004b08a0,6,133.4800,12.5200
0xdce817f009be265a9a8d605fd5d5a41cf34f0226078a3f039d2d1ba3aeb5bf0b,10,133.2798,79.1202
0xeeff20ced75ad4cb85f3ff09ca77307fc8b2d28e3ab6dc2dd03c69355eb94bdb,11,124.3423,193.5247


In [55]:
(
    buy_imp_test[buy_imp_test['condition_id'] == '0x18d19ee1593c5f758c9b652fae58b0ea98e8b28f3cc6c6ce42816dd032f9c8a7']
)

,follower_wallet,condition_id,outcome,follower_dt,pnl,notional,copyable_pnl,copyable_notional,leader_wallet,leader_dt,time_delta_seconds
1975,0x919698b19427cbe6945b0dc823f2d9e126a4d934,0x18d19ee1593c5f758c9b652fae58b0ea98e8b28f3cc6...,No,2026-07-09 08:05:49+00:00,1.9350,75.4650,0.0000,0.0000,0x85d22a973e817f87afe0a7fd346a39823cce7479,2026-07-09 08:04:55+00:00,54.0000
